# Dataset inspection

Adapted from origin commit `77947cd`. Run the **Load data** cell first, then **Sample plots** to view one original goal and its six chronological history images. Other sections remain inspection tasks, not implemented analyses.

Run only on Perlmutter, using the existing `pytorch/2.13.0` Python environment. The equivalent display snippets worked in owner-run notebook execution after loader tests at `9221135`. This shared notebook stores code only: keep sensitive records, images, plots, and outputs on Perlmutter and clear outputs before committing. See `docs/LOADER.md` for usage and `docs/grp2/PROGRESS.md` for results and limits.


## Load data

Reuse `GuidewireDataset` from `guidewire/data/dataset.py` with an explicit `split_position` and `partition`. It returns original Pillow images and metadata, without preprocessing or batching. Split position is not the stored fold value. Choosing an inspection partition does not establish the evaluation protocol.

In [ ]:
import os
from pathlib import Path
from IPython.display import display
from guidewire.data.dataset import GuidewireDataset

data_dir = Path(os.environ["PSCRATCH"]) / "CS5470_FinalProject/shorthorizon2"
# Split position 0 is an inspection example, not an evaluation decision.
cases = GuidewireDataset(data_dir, split_position=0, partition="train")
case = cases[0]

# The goal is a separate reference, not another chronological history frame.
print("Goal reference")
display(case["goal"])


## Class balance

Already confirmed in owner-run checks: 416 cases; ADVANCE = 1 (259), RETRACT = 0 (157). These are operator-action proxies, not verified optimal actions. Reuse these facts; partition-level summaries can be added when needed for experiments.

## Frame timing

Six `history_offsets_s` values align with the six chronological history images and six `depth_relative_mm` values. Both sequences end at zero; time spacing can vary. Explore timing distributions on Perlmutter if needed. Whether models use time remains undecided.

## Frames per decision point

Confirmed structure: one separate target-marked goal reference plus six chronological history images. The goal is not a seventh history step. The raw reader returns `goal` and `history` separately; do not collapse their roles when preparing model inputs.

## Decision points per episode

Confirmed aggregate: 416 decision points from 120 episodes. Use the supplied `episode` grouping; correspondence to a clinical procedure is not independently established. Within-split episode separation has already passed owner-run checks. Episode-size analysis remains optional.

## Image resolution and wire width

Original dimensions are 1065 × 1050. PNG header checks passed; full decoding and visual correctness remain pending. Inspect wire and target visibility on Perlmutter before adopting any resize, crop, padding, color conversion, or normalization. No model input resolution has been selected.

## Sample plots

After **Load data**, run the cell below to display six history images from oldest to newest with their supplied depths and time offsets. No resizing, normalization, augmentation, or model code is applied. Successful display does not establish semantic correctness; assess wire/target visibility and image/metadata meaning on Perlmutter.


In [ ]:
for i, image in enumerate(case["history"]):
    depth = case["depth_relative_mm"][i]
    time = case["history_offsets_s"][i]
    print(f"Frame {i + 1}: depth={depth} mm, time={time} s")
    display(image)


## Group 2 — ViT feature diagnostics (Perlmutter only)

Follow `docs/grp2/VIT_DIAGNOSTICS.md`. These cells inspect the existing frozen encoder's 384-value whole-image summaries; they do not train or establish destination understanding. Keep all previews, coordinates, features, and notebook outputs on Perlmutter. No files are saved by these cells.

Use the existing scratch venv containing timm in an allocated compute session or appropriately allocated notebook. A kernel does not itself allocate a GPU. Preserve your remote notebook before transferring any code; no Git handoff is implied. Run **Load data** above first. Its `cases` variable must refer to a training partition.


### V1 — Choose one training case and prepare images

Start with case index 0. Inspect originals versus prepared previews: wire tip, marker and image edges must remain visible. The 518×518 full-view trial is already implemented; this is targeted visual inspection, not repeating the CPU extraction milestone. Later change the index to inspect a few varied training cases; this is not an evaluation split decision.


In [ ]:
import torch
from guidewire.data.preprocessing import DINOv2Preprocessor
from guidewire.encoders.vit import DINOv2Encoder

assert cases.partition == "train", "Use training cases for exploratory diagnostics"
diag_case_index = 0
diag_case = cases[diag_case_index]
diag_prepare = DINOv2Preprocessor()
diag_prepared = diag_prepare.prepare_case(diag_case)
print("Goal: original, then model input")
display(diag_case["goal"], diag_prepared["previews"]["goal"])
print("Latest history: original, then model input")
display(diag_case["history"][5], diag_prepared["previews"]["history"][5])


### V2 — Load encoder once and define feature distances

Public weights may download; images are processed locally. Reuse `diag_encoder` when rerunning. This uses the same backbone feature/head calls as `guidewire/encoders/vit.py` under inference mode. Relative L2 measures change relative to the original vector's length; cosine distance measures a change in vector direction. Neither is accuracy, and neither has a validated pass/fail threshold here. Depth/time and labels do not enter the ViT.


In [ ]:
if "diag_encoder" not in globals():
    diag_encoder = DINOv2Encoder(
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
print("Diagnostic device:", diag_encoder.device)

@torch.inference_mode()
def diag_describe(image):
    assert image.shape == (3, 518, 518)
    assert image.dtype == torch.float32 and torch.isfinite(image).all()
    diag_encoder.backbone.eval()
    tokens = diag_encoder.backbone.forward_features(
        image.unsqueeze(0).to(diag_encoder.device)
    )
    vector = diag_encoder.backbone.forward_head(tokens, pre_logits=True)[0]
    assert vector.shape == (384,) and torch.isfinite(vector).all()
    return vector.cpu()

def diag_compare(original, changed):
    return {
        "relative_L2": float(torch.linalg.vector_norm(original - changed)
            / torch.linalg.vector_norm(original).clamp_min(1e-8)),
        "cosine_distance": float(1 - torch.nn.functional.cosine_similarity(
            original.unsqueeze(0), changed.unsqueeze(0))),
    }


### V3 — Inspect the marked/unmarked pair before measuring

Read the corresponding `clean_paths[0]` using the loader's existing path-safety checks. Check that the only visible difference is the annotation. The amplified difference preview helps locate other changes; it is not a model explanation. If the underlying scenes differ, do not interpret this as a marker-only comparison.


In [ ]:
from PIL import ImageChops, ImageEnhance

diag_record = cases.records[diag_case["row_index"]]
diag_raw_goal = cases._read_image(
    diag_record["clean_paths"][0], diag_case["row_index"]
)
assert diag_raw_goal.size == diag_case["goal"].size
diag_marked = diag_prepare.prepare_image(diag_case["goal"])
diag_clean = diag_prepare.prepare_image(diag_raw_goal)
print("Marked, unmarked, amplified pixel difference")
display(diag_marked["preview"], diag_clean["preview"])
diag_diff = ImageChops.difference(diag_marked["preview"], diag_clean["preview"])
display(ImageEnhance.Brightness(diag_diff).enhance(4))
diag_pair_confirmed = False  # Set True below only after visual inspection.


In [ ]:
# Change to True only if the pair shows the same scene apart from the marker.
diag_pair_confirmed = False
assert diag_pair_confirmed, "Inspect the pair above before interpreting marker removal"
diag_goal_features = diag_describe(diag_marked["tensor"])
print("Repeat unchanged image:", diag_compare(
    diag_goal_features, diag_describe(diag_marked["tensor"])))
print("Remove marker:", diag_compare(
    diag_goal_features, diag_describe(diag_clean["tensor"])))


### V4 — Choose wire-tip and control regions

Start with the latest history frame (`5`); indices 0–5 are oldest to newest. Use the grid to read coordinates `(x, y)` measured from the top-left of the prepared image. Choose one center on the visible tip and three controls away from the wire, within the actual image rather than padding. Prefer similar local brightness/texture where possible. If you cannot identify the tip confidently, stop this probe instead of guessing. Matplotlib is needed for this display.

**What control 1, 2 and 3 mean:** these are three comparison regions you choose that do not contain the wire, not extra models. Cover each separately with the same size square used for the wire tip. If covering any region changes the features substantially, a wire-tip change alone tells us little. A consistently larger change at the tip supports sensitivity to that region. Three comparisons reduce reliance on one unusual region; three is not a model requirement.

**How to choose coordinates:** in V4, read x horizontally (increasing right) and y vertically (increasing down), starting at (0, 0) in the top-left. Use the displayed 518×518 prepared image, not the original. Center the wire square on its visible tip. Choose three distinct non-wire regions with reasonably similar brightness/texture; avoid text, borders, padding and choosing only blank background. Keep centers at least 14 pixels inside the actual image so the entire 28×28 square fits. Replace each `None` in V5 with a numeric `(x, y)` pair. Inspect the covered-image previews: the tip square must cover the tip and comparison squares must avoid the wire. Adjust coordinates if necessary; skip a case if you cannot confidently identify the tip.


In [ ]:
import matplotlib.pyplot as plt

diag_frame_index = 5
assert 0 <= diag_frame_index < 6
diag_preview = diag_prepared["previews"]["history"][diag_frame_index]
fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(diag_preview)
ax.set_xticks(range(0, 518, 28))
ax.set_yticks(range(0, 518, 28))
ax.grid(alpha=0.3)
ax.set_title("Prepared history image: coordinates (x, y)")
plt.show()


### V5 — Cover equal-size regions and measure feature changes

Replace each `None` with your observed `(x, y)` center. Every square is 28×28 prepared-image pixels (two ViT patches wide). Masking fills normalized tensors with zero: the ImageNet mean color, not black. Previews show the same fill rounded to 8-bit RGB. Check that the edits cover the intended regions. Larger changes indicate sensitivity, not localization accuracy. Patch edges, anatomy and background can affect the result too.


In [ ]:
diag_centers = {
    "wire_tip": None,
    "control_1": None,
    "control_2": None,
    "control_3": None,
}
assert all(point is not None for point in diag_centers.values()),     "Replace None with observed (x, y) centers from V4"
diag_patch_size = 28
diag_image = diag_prepared["history"][diag_frame_index]
diag_baseline = diag_describe(diag_image)
diag_geometry = diag_prepared["preparation"]["history"][diag_frame_index]
diag_pad_left, diag_pad_top, _, _ = diag_geometry["padding_ltrb"]
diag_width, diag_height = diag_geometry["resized_size_wh"]
diag_results = {}
for name, (x, y) in diag_centers.items():
    left = int(x) - diag_patch_size // 2
    top = int(y) - diag_patch_size // 2
    right, bottom = left + diag_patch_size, top + diag_patch_size
    assert diag_pad_left <= left < right <= diag_pad_left + diag_width
    assert diag_pad_top <= top < bottom <= diag_pad_top + diag_height
    changed = diag_image.clone()
    changed[:, top:bottom, left:right] = 0
    diag_results[name] = diag_compare(diag_baseline, diag_describe(changed))
    shown = diag_preview.copy()
    shown.paste(diag_prepare.padding_rgb, (left, top, right, bottom))
    print(name, diag_results[name])
    display(shown)


### V6 — Interpret and record, then repeat cautiously

The unchanged-image comparison should be approximately zero. A larger marked/unmarked difference indicates marker sensitivity, not destination/location understanding. Compare the wire edit with all three controls: consistent stronger changes support sensitivity to the tip region, but are not proof of wire detection. Equal or weaker changes are inconclusive.

Repeat V1 and V3–V5 for 3–5 varied training cases, selecting new coordinates each time; reuse the encoder. Report only permitted aggregate distances/device/software versions and qualitative conclusions in the group log. Keep case identifiers, coordinates, images, features and detailed notebook results on Perlmutter. No arbitrary distance threshold, action-accuracy claim, or automatic freeze/fine-tune decision follows.

If useful sensitivity is observed, discuss the frozen ViT–GRU input contract next. If ambiguous, inspect preparation and consider whether whole-image pooling loses spatial detail before expanding experiments. Moving the goal marker or inspecting patch tokens is a possible later probe, not implemented here. Clear outputs only in a separate shareable copy after preserving the remote notebook; never overwrite the sole copy of your remote work.


## Group 1 - wire preprocessing inspection (Perlmutter only)

These cells set two `WirePreprocessor` parameters by looking at training images, without labels:

- `tophat_size`: the smallest window where the whole wire lights up but blob-shaped anatomy does not;
- `crop_size`: the smallest crop that contains the wire tip in at least 90% of cases.

Run **Load data** above first; these cells use its `cases`, which must be split position 0, `train`.
Never use validation or test cases here.
Samples are seeded, so reruns show the same cases.
Keep every image, plot and tally on Perlmutter, and clear outputs before committing.

### W1 - Channel check

For a seeded sample of 20 training cases, each row shows the newest history frame's 256 crop around `goal_xy`, then the black top-hat at windows 7, 11 and 15, then the difference channel.

- **Top-hat:** the wire should be bright along its whole length and the background dark.
  Too small a window lights only the wire's edges or breaks it into pieces; too large a window also lights wider dark anatomy such as vessels or blobs.
  Pick the smallest window that lights the whole wire without lighting blobs.
- **Difference (median):** each frame minus the per-pixel median of the six history crops.
  Blue is darker than usual (where the dark wire newly is), red is brighter than usual (where it used to be).
  Static anatomy should be near white (zero).

In [ ]:
import math
import random
import matplotlib.pyplot as plt
from guidewire.data.preprocessing import MEAN, STD, WirePreprocessor

assert cases.split_position == 0 and cases.partition == "train", \
    "Run Load data with split_position=0 and partition='train'"
g1_tophat_sizes = (7, 11, 15)
# Default (0, 1) stats leave the top-hat and difference channels in 0-1 pixel units.
g1_prepares = {size: WirePreprocessor(256, "enhanced", tophat_size=size)
               for size in g1_tophat_sizes}
g1_channel_indices = sorted(random.Random(0).sample(range(len(cases)), 20))

for sample, index in enumerate(g1_channel_indices):
    g1_case = cases[index]
    newest = {size: prep.prepare_case(g1_case)["history"][5]
              for size, prep in g1_prepares.items()}
    raw = newest[11][0] * STD[0] + MEAN[0]
    difference = newest[11][2]
    # One top-hat scale for all sizes so their brightness is comparable.
    tophat_max = max(float(newest[size][1].max()) for size in g1_tophat_sizes) or 1.0
    difference_max = float(difference.abs().max()) or 1.0
    fig, axes = plt.subplots(1, 5, figsize=(20, 4.5))
    axes[0].imshow(raw, cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("raw crop (newest)")
    for ax, size in zip(axes[1:4], g1_tophat_sizes):
        ax.imshow(newest[size][1], cmap="gray", vmin=0, vmax=tophat_max)
        ax.set_title(f"top-hat {size}")
    axes[4].imshow(difference, cmap="RdBu_r", vmin=-difference_max, vmax=difference_max)
    axes[4].set_title("difference (median)")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(f"Channel check sample {sample + 1} of 20")
    plt.show()

### W2 - Crop-size boxes

A different seeded sample of 40 training cases shows the newest history frame at full resolution with four boxes centered on `goal_xy`, exactly where `WirePreprocessor` would crop: 192 red, 256 orange, 384 green, 512 blue.
Find the wire tip, then fill in the tally cell below with the smallest box that contains it, `"outside"` if none does, or leave `None` if you cannot see the tip.
Run the count cell after the tally; it prints only aggregate counts.

In [ ]:
from PIL import ImageDraw

g1_box_sizes = {192: (255, 0, 0), 256: (255, 165, 0), 384: (0, 200, 0), 512: (0, 128, 255)}
g1_box_indices = sorted(random.Random(1).sample(range(len(cases)), 40))

for sample, index in enumerate(g1_box_indices):
    g1_case = cases[index]
    shown = g1_case["history"][5].convert("RGB")
    draw = ImageDraw.Draw(shown)
    # Same center as goal_crop: goal_xy rounded half up.
    cx, cy = (math.floor(value + 0.5) for value in g1_case["goal_xy"])
    for size, color in g1_box_sizes.items():
        left, top = cx - size // 2, cy - size // 2
        draw.rectangle((left, top, left + size - 1, top + size - 1), outline=color, width=2)
    print(f"Sample {sample}: 192 red, 256 orange, 384 green, 512 blue")
    display(shown)

In [ ]:
# For each sample number printed above, record the smallest box that contains
# the wire tip: 192, 256, 384, 512, or "outside". Leave None if the tip is not
# visible. The tally stays in this session; never commit it.
g1_tally = {sample: None for sample in range(40)}

In [ ]:
from collections import Counter

g1_seen = [value for value in g1_tally.values() if value is not None]
assert all(value in (*g1_box_sizes, "outside") for value in g1_seen), \
    "Use 192, 256, 384, 512 or 'outside'"
print("Tips recorded:", len(g1_seen), "of", len(g1_tally))
for size in g1_box_sizes:
    inside = sum(1 for value in g1_seen if value != "outside" and value <= size)
    print(f"Inside {size}: {inside}")
print("Outside 512:", Counter(g1_seen)["outside"])

# Optional: keep the tally under $PSCRATCH, never in the repository.
# import json
# g1_tally_dir = Path(os.environ["PSCRATCH"]) / "waldo-inspection"
# g1_tally_dir.mkdir(parents=True, exist_ok=True)
# (g1_tally_dir / "crop_tally.json").write_text(json.dumps(g1_tally))

### W3 - What to report

Report counts only, in `docs/grp1/PROGRESS.md` or the group chat:

- the chosen `tophat_size` (7, 11 or 15) and one sentence on why;
- from the W2 tally, how many of 40 tips fall inside 192, 256, 384 and 512, how many fall outside 512, and how many were not visible.

Never report case indices, coordinates, images or crops.
The crop rule from the tickets: pick the smallest size that contains the tip in at least 90% of visible cases.
If even 384 misses more than about 10%, the optional context view (`--context-size`) becomes worth testing.